<a href="https://colab.research.google.com/github/miray7yuce/quadcopter-rl-copilot/blob/main/notebooks/quadcopter_rl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q stable-baselines3 gymnasium
!pip uninstall -y -q jsbsim
!pip install -q jsbsim==1.2.4

import jsbsim
print(jsbsim.__version__)

1.2.4


In [ ]:
from google.colab import userdata
import os

USER  = "miray7yuce"
REPO  = "quadcopter-rl-copilot"
TOKEN = userdata.get('GH_TOKEN')

!git config --global user.email "miray7yuce@gmail.com"
!git config --global user.name "miray7yuce"

os.environ['REMOTE'] = f"https://{TOKEN}@github.com/{USER}/{REPO}.git"
!rm -rf /content/repo
!git clone -q $REMOTE /content/repo
!ls -a /content/repo

.   configs  .gitignore  README.md	   runs
..  .git     notebooks	 requirements.txt  src


In [ ]:
!curl -fsSl https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs
!sudo npm install -g @anthropic-ai/claude-code

2026-08-31 10:05:02 - Installing pre-requisites
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://deb.nodesource.com/node_20.x nodistro InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
30 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' do

In [ ]:
from google.colab import userdata
import os
os.environ["ANTHROPIC_API_KEY"]=userdata.get('tusaş')

In [ ]:
!pip freeze | grep -iE "^(jsbsim|stable-baselines3|gymnasium|torch|numpy)=" > /content/repo/requirements.txt
!cat /content/repo/requirements.txt

gymnasium==1.3.0
jsbsim==1.2.4
numpy==2.1.3


In [ ]:
import os, sys

BASE = "/content/repo"

for d in ["src/drone_rl/envs", "src/drone_rl/utils", "configs"]:
    os.makedirs(f"{BASE}/{d}", exist_ok=True)

for p in ["src/drone_rl", "src/drone_rl/envs", "src/drone_rl/utils"]:
    open(f"{BASE}/{p}/__init__.py", "a").close()

with open(f"{BASE}/.gitignore", "w") as f:
    f.write("__pycache__/\n*.zip\n*.pkl\nlogs/\nruns/\n.ipynb_checkpoints/\n")

sys.path.insert(0, f"{BASE}/src")

!find /content/repo -not -path '*/.git/*' -type f | sort

os.environ['PYTHONPATH'] = f"{BASE}/src"

/content/repo/configs/ppo_hover.yaml
/content/repo/.gitignore
/content/repo/notebooks/quadcopter_rl.ipynb
/content/repo/README.md
/content/repo/requirements.txt
/content/repo/runs/hover_v1/model.zip
/content/repo/runs/hover_v1/vecnormalize.pkl
/content/repo/src/drone_rl/envs/f450_env.py
/content/repo/src/drone_rl/envs/__init__.py
/content/repo/src/drone_rl/evaluate.py
/content/repo/src/drone_rl/__init__.py
/content/repo/src/drone_rl/train.py
/content/repo/src/drone_rl/utils/__init__.py
/content/repo/src/drone_rl/utils/units.py


In [ ]:
%%writefile /content/repo/src/drone_rl/utils/units.py
"""Birim donusumleri. JSBSim emperyal birim kullanir."""

FT2M = 0.3048
M2FT = 1.0 / FT2M

def ft_to_m(x):
    return x * FT2M

def m_to_ft(x):
    return x * M2FT

Overwriting /content/repo/src/drone_rl/utils/units.py


In [ ]:
%%writefile /content/repo/src/drone_rl/envs/f450_env.py
"""F450 quadcopter icin hover gorevi ortami."""

import numpy as np
import gymnasium as gym
from gymnasium import spaces
import jsbsim


class F450HoverEnv(gym.Env):
    """JSBSim F450 modeli uzerinde sabit irtifada durma (hover) gorevi.

    Tum sayisal ayarlar (hedef irtifa, hover gazi, odul agirliklari,
    crash esikleri) constructor parametreleridir ve
    configs/ppo_hover.yaml + drone_rl.config.load_config ile
    doldurulabilir. Parametre verilmezse, refactor-oncesi kodda
    hardcoded olan degerlerle AYNI varsayilanlar kullanilir.
    """

    metadata = {"render_modes": []}

    def __init__(
        self,
        target_altitude_ft=30.0,
        episode_seconds=20.0,
        physics_hz=240,
        control_hz=20,
        hover_throttle=0.420,
        throttle_range=0.25,
        reward_alt_weight=0.10,
        reward_tilt_weight=0.50,
        reward_spin_weight=0.10,
        reward_jerk_weight=0.05,
        crash_penalty=50.0,
        crash_min_alt_ft=1.0,
        crash_max_alt_offset_ft=60.0,
        crash_max_tilt_rad=1.0,
    ):
        super().__init__()

        self.action_space = spaces.Box(-1.0, 1.0, shape=(4,), dtype=np.float32)
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(13,), dtype=np.float32)

        self.target_altitude = target_altitude_ft
        self.physics_dt = 1.0 / physics_hz
        self.control_hz = control_hz
        self.substeps = int(physics_hz / control_hz)
        self.max_steps = int(episode_seconds * control_hz)

        self.hover_throttle = hover_throttle
        self.throttle_range = throttle_range

        self.reward_alt_weight = reward_alt_weight
        self.reward_tilt_weight = reward_tilt_weight
        self.reward_spin_weight = reward_spin_weight
        self.reward_jerk_weight = reward_jerk_weight
        self.crash_penalty = crash_penalty
        self.crash_min_alt_ft = crash_min_alt_ft
        self.crash_max_alt_offset_ft = crash_max_alt_offset_ft
        self.crash_max_tilt_rad = crash_max_tilt_rad

        self.fdm = jsbsim.FGFDMExec(None)
        self.fdm.set_debug_level(0)
        if not self.fdm.load_model("F450"):
            raise RuntimeError("F450 modeli yuklenemedi")
        self.fdm.set_dt(self.physics_dt)

        self.step_count = 0
        self.prev_action = np.zeros(4, dtype=np.float32)

    @property
    def control_dt(self):
        """Bir 'step()' cagrisinin temsil ettigi sure (saniye).
        evaluate.py gibi diger kodun control_hz'i elle tekrar
        hesaplamasina gerek kalmaz, dogrudan buradan okur."""
        return self.substeps * self.physics_dt

    def _apply_initial_conditions(self):
        h0 = self.target_altitude + self.np_random.uniform(-3.0, 3.0)
        self.fdm["ic/h-agl-ft"] = h0
        self.fdm["ic/u-fps"] = self.np_random.uniform(-1.0, 1.0)
        self.fdm["ic/v-fps"] = self.np_random.uniform(-1.0, 1.0)
        self.fdm["ic/w-fps"] = self.np_random.uniform(-1.0, 1.0)
        self.fdm["ic/phi-rad"] = self.np_random.uniform(-0.05, 0.05)
        self.fdm["ic/theta-rad"] = self.np_random.uniform(-0.05, 0.05)
        self.fdm["ic/psi-true-rad"] = 0.0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self._apply_initial_conditions()
        self.fdm.run_ic()

        for i in range(4):
            self.fdm[f"propulsion/engine[{i}]/set-running"] = 1
        self.fdm["fcs/ScasEngage"] = 0

        for i in range(4):
            self.fdm[f"fcs/throttle-cmd-norm[{i}]"] = self.hover_throttle

        self.step_count = 0
        self.prev_action = np.zeros(4, dtype=np.float32)

        return self._get_obs(), {}

    def _get_obs(self):
        f = self.fdm
        alt_err = (f["position/h-agl-ft"] - self.target_altitude) / 10.0
        hdot = f["velocities/h-dot-fps"] / 10.0
        u = f["velocities/u-fps"] / 10.0
        v = f["velocities/v-fps"] / 10.0
        roll = f["attitude/phi-rad"]
        pitch = f["attitude/theta-rad"]
        p = f["velocities/p-rad_sec"] / 5.0
        q = f["velocities/q-rad_sec"] / 5.0
        r = f["velocities/r-rad_sec"] / 5.0

        return np.array(
            [alt_err, hdot, u, v, roll, pitch, p, q, r, *self.prev_action],
            dtype=np.float32,
        )

    def _is_crashed(self):
        alt = self.fdm["position/h-agl-ft"]
        return (
            alt < self.crash_min_alt_ft
            or alt > self.target_altitude + self.crash_max_alt_offset_ft
            or abs(self.fdm["attitude/phi-rad"]) > self.crash_max_tilt_rad
            or abs(self.fdm["attitude/theta-rad"]) > self.crash_max_tilt_rad
        )

    def step(self, action):
        throttles = np.clip(
            self.hover_throttle + action * self.throttle_range, 0.0, 1.0
        )

        for _ in range(self.substeps):
            for i in range(4):
                self.fdm[f"fcs/throttle-cmd-norm[{i}]"] = float(throttles[i])
            self.fdm.run()

        self.step_count += 1
        obs = self._get_obs()

        alt_err_ft = abs(self.fdm["position/h-agl-ft"] - self.target_altitude)
        tilt = abs(self.fdm["attitude/phi-rad"]) + abs(self.fdm["attitude/theta-rad"])
        spin = abs(self.fdm["velocities/p-rad_sec"]) + abs(self.fdm["velocities/q-rad_sec"])
        jerk = float(np.sum(np.abs(action - self.prev_action)))

        reward = (
            1.0
            - self.reward_alt_weight * alt_err_ft
            - self.reward_tilt_weight * tilt
            - self.reward_spin_weight * spin
            - self.reward_jerk_weight * jerk
        )

        crashed = self._is_crashed()
        if crashed:
            reward -= self.crash_penalty

        self.prev_action = action.copy()

        terminated = bool(crashed)
        truncated = bool(self.step_count >= self.max_steps)

        return obs, float(reward), terminated, truncated, {}

Overwriting /content/repo/src/drone_rl/envs/f450_env.py


In [ ]:
%%writefile /content/repo/src/drone_rl/train.py
"""F450 hover gorevi icin PPO egitimi + EvalCallback."""

import argparse
from pathlib import Path

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback, EvalCallback

from drone_rl.config import load_config
from drone_rl.env_factory import make_training_vec_env


class SaveVecNormalizeCallback(BaseCallback):
    """EvalCallback her yeni en iyi modeli bulduğunda, o anki
    VecNormalize istatistiklerini de best_model klasörüne kaydeder.
    Bunsuz best_model.zip yüklendiğinde yanlış olceklenmis
    gozlemlerle calisir (best_model aninda kaydedilen istatistikler,
    egitim sonunda kaydedilenle ayni olmayabilir)."""

    def __init__(self, save_path: Path):
        super().__init__()
        self.save_path = Path(save_path)

    def _on_step(self) -> bool:
        vec_normalize = self.model.get_vec_normalize_env()
        if vec_normalize is not None:
            self.save_path.mkdir(parents=True, exist_ok=True)
            vec_normalize.save(str(self.save_path / "vecnormalize_best.pkl"))
        return True


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", type=str, default=None,
                     help="configs/ppo_hover.yaml gibi bir config dosyasi; "
                          "verilmezse kod-ici varsayilanlar kullanilir "
                          "(refactor oncesi hardcoded degerlerle ayni)")
    ap.add_argument("--timesteps", type=int, default=None,
                     help="Verilirse config'teki train.timesteps'i gecersiz kilar")
    ap.add_argument("--n-envs", type=int, default=None,
                     help="Verilirse config'teki train.n_envs'i gecersiz kilar")
    ap.add_argument("--out", type=str, default="/content/runs/hover")
    ap.add_argument("--eval-freq", type=int, default=10000)
    args = ap.parse_args()

    cfg = load_config(args.config)
    timesteps = args.timesteps if args.timesteps is not None else cfg.train.timesteps
    n_envs = args.n_envs if args.n_envs is not None else cfg.train.n_envs

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    # Egitim Ortami
    venv = make_training_vec_env(cfg.env, n_envs=n_envs, training=True, norm_reward=True)

    # Degerlendirme Ortami (Callback icin) - normalizasyon istatistikleri
    # sync_envs_normalization ile EvalCallback tarafindan egitim
    # ortamindan otomatik kopyalanir.
    eval_env = make_training_vec_env(cfg.env, n_envs=1, training=False, norm_reward=False)

    model = PPO(
        cfg.ppo.policy, venv,
        n_steps=cfg.ppo.n_steps, batch_size=cfg.ppo.batch_size, n_epochs=cfg.ppo.n_epochs,
        gamma=cfg.ppo.gamma, gae_lambda=cfg.ppo.gae_lambda, clip_range=cfg.ppo.clip_range,
        learning_rate=cfg.ppo.learning_rate, ent_coef=cfg.ppo.ent_coef,
        verbose=1, device="cpu",
        tensorboard_log=str(out / "tb"),
    )

    # 1. Checkpoint Callback: Periyodik kayit
    ckpt_cb = CheckpointCallback(
        save_freq=max(20_000 // n_envs, 1),
        save_path=str(out / "ckpt"),
        name_prefix="ppo",
    )

    # 2. Eval Callback: En iyi modeli bulma ve test
    best_model_path = out / "best_model"
    save_vecnorm_cb = SaveVecNormalizeCallback(best_model_path)

    eval_cb = EvalCallback(
        eval_env,
        best_model_save_path=str(best_model_path),
        callback_on_new_best=save_vecnorm_cb,
        log_path=str(out / "logs"),
        eval_freq=max(args.eval_freq // n_envs, 1),
        deterministic=True,
        render=False,
    )

    model.learn(total_timesteps=timesteps, callback=[ckpt_cb, eval_cb])

    model.save(out / "model_final")
    venv.save(str(out / "vecnormalize.pkl"))
    print("Egitim tamamlandi ve kaydedildi:", out)
    print("  Son model      :", out / "model_final.zip", "+", out / "vecnormalize.pkl")
    print("  En iyi model   :", best_model_path / "best_model.zip", "+", best_model_path / "vecnormalize_best.pkl")


if __name__ == "__main__":
    main()

Overwriting /content/repo/src/drone_rl/train.py


In [ ]:
# EvalCallback ile güncellenmiş yeni eğitimi başlat
!cd /content/repo/src && python -m drone_rl.train \
    --timesteps 300000 \
    --n-envs 4 \
    --eval-freq 10000 \
    --out /content/runs/hover_v2

2026-08-31 12:42:02.003947: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-31 12:42:02.278070: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


     JSBSim Flight Dynamics Model v1.2.4 Feb  7 2026 11:12:49
            [JSBSim-ML v2.0]

JSBSim startup beginning ...


YOU HAVE AN INCOMPATIBLE CFG FILE FOR THIS AIRCRAFT.

In [ ]:
#training sürecini başlatır 300.000 step
!cd /content/repo/src && python -m drone_rl.train --timesteps 300000 --n-envs 4 --out /content/runs/hover_v1

2026-08-31 08:00:47.598010: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-31 08:00:47.692590: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


     JSBSim Flight Dynamics Model v1.2.4 Feb  7 2026 11:12:49
            [JSBSim-ML v2.0]

JSBSim startup beginning ...


YOU HAVE AN INCOMPATIBLE CFG FILE FOR THIS AIRCRAFT.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/runs

In [ ]:
%%writefile /content/repo/src/drone_rl/evaluate.py
"""Egitilmis politikayi calistir; CSV telemetri ve/veya gercek ACMI
(Tacview) dosyasi olarak kaydet."""

import argparse
from pathlib import Path
import numpy as np
import pandas as pd
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import VecNormalize

from drone_rl.config import load_config
from drone_rl.env_factory import make_eval_vec_env
from drone_rl.utils.units import ft_to_m
from drone_rl.acmi_writer import ACMIWriter


def resolve_model_paths(run: Path, use_best: bool):
    """--use-best bayragina gore yuklenecek model + VecNormalize dosya
    yollarini dondurur. train.py'nin ciktisiyla birebir eslesmesi gereken
    tek yer burasi - isimler degisirse sadece burasi guncellenir."""
    if use_best:
        return run / "best_model" / "best_model", run / "best_model" / "vecnormalize_best.pkl"
    return run / "model_final", run / "vecnormalize.pkl"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", type=str, default=None,
                     help="Egitimde kullanilan configs/ppo_hover.yaml. "
                          "Ortam parametrelerinin (hover_throttle, control_hz vb.) "
                          "egitimle AYNI olmasi icin, egitimde --config kullandiysan "
                          "burada da ayni dosyayi vermelisin.")
    ap.add_argument("--run", type=str, default="/content/repo/runs/hover_v1")
    ap.add_argument("--episodes", type=int, default=3)
    ap.add_argument("--output", type=str, default="/content/telemetry.csv",
                     help="CSV telemetri ciktisi")
    ap.add_argument("--acmi-output", type=str, default=None,
                     help="Verilirse, gercek Tacview .acmi dosyasi da bu yola yazilir")
    ap.add_argument(
        "--use-best", action="store_true",
        help="model_final yerine best_model + vecnormalize_best kullan"
    )
    args = ap.parse_args()

    cfg = load_config(args.config)
    run = Path(args.run)
    model_path, vecnorm_path = resolve_model_paths(run, args.use_best)

    if not vecnorm_path.exists():
        raise FileNotFoundError(
            f"VecNormalize dosyasi bulunamadi: {vecnorm_path}\n"
            "train.py'nin ciktisi ile bu betigin bekledigi dosya adlari "
            "arasinda bir uyumsuzluk olabilir; --run yolunu kontrol edin."
        )

    venv = make_eval_vec_env(cfg.env)
    venv = VecNormalize.load(str(vecnorm_path), venv)
    venv.training = False
    venv.norm_reward = False

    model = PPO.load(str(model_path), device="cpu")
    raw = venv.envs[0]
    control_dt = raw.control_dt  # f450_env.py'den; control_hz'i elle tekrarlamiyoruz

    all_telemetry = []
    acmi = ACMIWriter(name="F450", obj_type="Air+Rotorcraft+UAV", color="Blue") \
        if args.acmi_output else None

    global_t = 0.0  # ACMI icin episode'lar arasi kesintisiz artan zaman

    for ep in range(args.episodes):
        obs = venv.reset()
        t = 0.0

        while True:
            action, _ = model.predict(obs, deterministic=True)
            obs, _, done, _ = venv.step(action)

            fdm = raw.fdm
            alt_ft = float(fdm["position/h-agl-ft"])
            roll_rad = float(fdm["attitude/phi-rad"])
            pitch_rad = float(fdm["attitude/theta-rad"])
            yaw_rad = float(fdm["attitude/psi-rad"])

            data = {
                "timestamp": round(t, 4),
                "episode": ep,
                "alt_ft": round(alt_ft, 4),
                "roll_rad": round(roll_rad, 6),
                "pitch_rad": round(pitch_rad, 6),
                "yaw_rad": round(yaw_rad, 6),
                "alt_err_ft": round(abs(alt_ft - raw.target_altitude), 4),
            }
            all_telemetry.append(data)

            if acmi is not None:
                lat_deg = float(fdm["position/lat-geod-deg"])
                lon_deg = float(fdm["position/long-gc-deg"])
                alt_m = ft_to_m(float(fdm["position/h-sl-ft"]))
                acmi.add_frame(
                    t=global_t,
                    lon_deg=lon_deg,
                    lat_deg=lat_deg,
                    alt_m=alt_m,
                    roll_deg=np.degrees(roll_rad),
                    pitch_deg=np.degrees(pitch_rad),
                    yaw_deg=np.degrees(yaw_rad),
                )

            t += control_dt
            global_t += control_dt
            if done[0]:
                break  # episode length

    df = pd.DataFrame(all_telemetry)
    df.to_csv(args.output, index=False, header=True)
    print(f"CSV telemetri kaydedildi: {args.output}")
    print(df.head())

    if acmi is not None:
        saved_path = acmi.save(args.acmi_output)
        print(f"ACMI (Tacview) dosyasi kaydedildi: {saved_path}")
        print(
            "Not: episode'lar arasi zaman kesintisiz devam ediyor "
            "(reset aninda arac konumda 'atlar' - bu normaldir, "
            "cunku her episode farkli baslangic irtifasindan basliyor)."
        )


if __name__ == '__main__':
    main()

Overwriting /content/repo/src/drone_rl/evaluate.py


In [ ]:
#gitignore oluşturur ve günceller
with open("/content/repo/.gitignore", "w") as f:
    f.write("__pycache__/\n.ipynb_checkpoints/\nruns/*/tb/\nruns/*/ckpt/\n")
!cat /content/repo/.gitignore

__pycache__/
.ipynb_checkpoints/
runs/*/tb/
runs/*/ckpt/


In [ ]:
# Egitilen modeli test etmek ve sonuclari kaydetmek icin
!cd /content/repo/src && python -m drone_rl.evaluate --run /content/repo/runs/hover_v1 --output /content/iz.csv

2026-08-31 11:27:07.015621: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-31 11:27:07.087577: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


     JSBSim Flight Dynamics Model v1.2.4 Feb  7 2026 11:12:49
            [JSBSim-ML v2.0]

JSBSim startup beginning ...


YOU HAVE AN INCOMPATIBLE CFG FILE FOR THIS AIRCRAFT.

In [ ]:
%%writefile /content/repo/configs/ppo_hover.yaml
env:
  target_altitude_ft: 30.0
  episode_seconds: 20.0
  control_hz: 20
  physics_hz: 240
  hover_throttle: 0.410
  throttle_range: 0.25
ppo:
  policy: MlpPolicy
  n_steps: 1024
  batch_size: 256
  n_epochs: 10
  gamma: 0.99
  gae_lambda: 0.95
  clip_range: 0.2
  learning_rate: 0.0003
  ent_coef: 0.0
train:
  timesteps: 300000
  n_envs: 4

Overwriting /content/repo/configs/ppo_hover.yaml


In [ ]:
readme = """# quadcopter-rl-copilot

JSBSim F450 quadcopter modeli uzerinde PPO ile hover kontrolu.

## Sonuc (hover_v1)

300.000 adim egitim sonrasi, 5 degerlendirme episode'unda:

- Episode uzunlugu: 400/400 (hic dusme yok)
- Hedef irtifadan ortalama sapma: 0.02 - 0.06 ft
- Ortalama egilme: 0.02 - 0.08 rad

## Kurulum (Colab)

    !pip install -q stable-baselines3 gymnasium
    !pip uninstall -y -q jsbsim
    !pip install -q jsbsim==1.2.4

Repoyu klonladiktan sonra src dizinini Python yoluna ekle:

    import os, sys
    sys.path.insert(0, "/content/repo/src")
    os.environ["PYTHONPATH"] = "/content/repo/src"

## Kullanim

Egitim:

    cd src && python -m drone_rl.train --timesteps 300000 --n-envs 4 --out ../runs/hover_v2

Degerlendirme:

    cd src && python -m drone_rl.evaluate --run ../runs/hover_v1 --csv /content/iz.csv

## Yapi

- src/drone_rl/envs/f450_env.py - Gymnasium ortami
- src/drone_rl/train.py - PPO egitimi
- src/drone_rl/evaluate.py - egitilmis politikanin olculmesi
- configs/ppo_hover.yaml - kullanilan ayarlar
- runs/hover_v1/ - egitilmis model ve normalizasyon istatistikleri
- notebooks/quadcopter_rl.ipynb - Colab calisma defteri

## Notlar

- JSBSim emperyal birim kullanir (ft, lbs, fps).
- Hover gazi 0.410 olarak olculdu. Aksiyon bu deger etrafinda +-0.25
  araliginda olceklenir, boylece sifir aksiyon "asili kal" anlamina gelir.
- vecnormalize.pkl model ile birlikte yuklenmelidir, aksi halde politika
  yanlis olcekli gozlem alir ve calismaz.
- F450 XML'i yuklenirken "version 3.0" uyarisi verir; zararsizdir.
"""

with open("/content/repo/README.md", "w") as f:
    f.write(readme)

print(open("/content/repo/README.md").read()[:300])

# quadcopter-rl-copilot

JSBSim F450 quadcopter modeli uzerinde PPO ile hover kontrolu.

## Sonuc (hover_v1)

300.000 adim egitim sonrasi, 5 degerlendirme episode'unda:

- Episode uzunlugu: 400/400 (hic dusme yok)
- Hedef irtifadan ortalama sapma: 0.02 - 0.06 ft
- Ortalama egilme: 0.02 - 0.08 rad

#


In [ ]:
%%writefile /content/repo/src/drone_rl/acmi_writer.py
"""Tacview ACMI (.acmi) format yazici - basit tek-obje coklu-episode kaydi.

ACMI format referansi: https://www.tacview.net/documentation/acmi/en/

Bu writer sadece bu proje icin gerekli minimum alt kumeyi destekler:
tek bir hava araci objesi, zaman serisi konum/aci guncellemeleri.
Coklu obje, olay (event) kayitlari, veya ek property'ler (hiz, RPM vb.)
desteklenmiyor - ihtiyac olursa genisletilebilir.

Beklenen birimler (ACMI standardi):
- lon_deg, lat_deg : derece (WGS84)
- alt_m            : metre (deniz seviyesinden veya yerden - tutarli olmasi yeterli)
- roll_deg, pitch_deg, yaw_deg : derece

JSBSim ft ve radyan kullandigi icin cagiran kod bu donusumu
(units.ft_to_m, np.degrees) yapmali; bu siniftan once cagirilmalidir.
"""

from pathlib import Path
from datetime import datetime, timezone


class ACMIWriter:
    def __init__(self, object_id=1, name="F450", obj_type="Air+Rotorcraft+UAV", color="Blue"):
        self.object_id = object_id
        self.name = name
        self.obj_type = obj_type
        self.color = color
        self._lines = []
        self._header_written = False
        self._object_declared = False
        self._last_frame_time = None

    def _write_header(self):
        now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
        self._lines.append("FileType=text/acmi/tacview")
        self._lines.append("FileVersion=2.2")
        self._lines.append(f"0,ReferenceTime={now}")
        self._header_written = True

    def add_frame(self, t, lon_deg, lat_deg, alt_m, roll_deg, pitch_deg, yaw_deg):
        """Bir zaman anindaki obje durumunu ekler.

        t: saniye cinsinden, dosya boyunca MONOTONIK ARTAN olmali
           (birden fazla episode varsa t'yi sifirlama, devam ettir).
        """
        if not self._header_written:
            self._write_header()

        # Ayni t icin tekrar frame acmayalim (float hassasiyeti icin yuvarla)
        t_rounded = round(t, 2)
        if self._last_frame_time != t_rounded:
            self._lines.append(f"#{t_rounded:.2f}")
            self._last_frame_time = t_rounded

        obj_line = (
            f"{self.object_id:x},T={lon_deg:.7f}|{lat_deg:.7f}|{alt_m:.2f}|"
            f"{roll_deg:.2f}|{pitch_deg:.2f}|{yaw_deg:.2f}"
        )
        if not self._object_declared:
            obj_line += f",Name={self.name},Type={self.obj_type},Color={self.color}"
            self._object_declared = True

        self._lines.append(obj_line)

    def save(self, path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            f.write("\n".join(self._lines) + "\n")
        return path


In [ ]:
%%writefile /content/repo/src/drone_rl/config.py
"""Merkezi config yukleme. YAML dosyasindaki degerleri dataclass'lara donusturur.

configs/ppo_hover.yaml gibi bir dosyayi okuyup Config nesnesine cevirir.
--config verilmezse (path=None), kod-ici varsayilan degerlerle bir Config
dondurulur - bu varsayilanlar, refactor oncesi f450_env.py/train.py'de
hardcoded olan degerlerle AYNIDIR, yani --config kullanilmadigi surece
davranis degismez.
"""

from dataclasses import dataclass, field
from typing import Optional
import yaml


@dataclass
class EnvConfig:
    target_altitude_ft: float = 30.0
    episode_seconds: float = 20.0
    control_hz: int = 20
    physics_hz: int = 240
    hover_throttle: float = 0.420
    throttle_range: float = 0.25
    reward_alt_weight: float = 0.10
    reward_tilt_weight: float = 0.50
    reward_spin_weight: float = 0.10
    reward_jerk_weight: float = 0.05
    crash_penalty: float = 50.0
    crash_min_alt_ft: float = 1.0
    crash_max_alt_offset_ft: float = 60.0
    crash_max_tilt_rad: float = 1.0


@dataclass
class PPOConfig:
    policy: str = "MlpPolicy"
    n_steps: int = 1024
    batch_size: int = 256
    n_epochs: int = 10
    gamma: float = 0.99
    gae_lambda: float = 0.95
    clip_range: float = 0.2
    learning_rate: float = 3e-4
    ent_coef: float = 0.0


@dataclass
class TrainConfig:
    timesteps: int = 300_000
    n_envs: int = 4


@dataclass
class Config:
    env: EnvConfig = field(default_factory=EnvConfig)
    ppo: PPOConfig = field(default_factory=PPOConfig)
    train: TrainConfig = field(default_factory=TrainConfig)


def load_config(path: Optional[str]) -> Config:
    """Verilen yaml dosyasini okuyup Config nesnesine donusturur.
    path None ise, tamamen varsayilan degerlerle bir Config doner
    (eski hardcoded davranisla ayni)."""
    if path is None:
        return Config()

    with open(path, "r") as f:
        raw = yaml.safe_load(f) or {}

    return Config(
        env=EnvConfig(**raw.get("env", {})),
        ppo=PPOConfig(**raw.get("ppo", {})),
        train=TrainConfig(**raw.get("train", {})),
    )

In [ ]:
%%writefile /content/repo/src/drone_rl/env_factory.py
"""Egitim ve degerlendirme icin ortak F450HoverEnv/VecEnv kurulum yardimcilari.

train.py ve evaluate.py'de tekrarlanan ortam kurulum mantigini
tekillestiriyoruz.
"""

from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from drone_rl.envs.f450_env import F450HoverEnv
from drone_rl.config import EnvConfig


def make_env(env_config: EnvConfig) -> F450HoverEnv:
    """Tek bir F450HoverEnv olusturur (Monitor sarmadan)."""
    return F450HoverEnv(
        target_altitude_ft=env_config.target_altitude_ft,
        episode_seconds=env_config.episode_seconds,
        physics_hz=env_config.physics_hz,
        control_hz=env_config.control_hz,
        hover_throttle=env_config.hover_throttle,
        throttle_range=env_config.throttle_range,
        reward_alt_weight=env_config.reward_alt_weight,
        reward_tilt_weight=env_config.reward_tilt_weight,
        reward_spin_weight=env_config.reward_spin_weight,
        reward_jerk_weight=env_config.reward_jerk_weight,
        crash_penalty=env_config.crash_penalty,
        crash_min_alt_ft=env_config.crash_min_alt_ft,
        crash_max_alt_offset_ft=env_config.crash_max_alt_offset_ft,
        crash_max_tilt_rad=env_config.crash_max_tilt_rad,
    )


def make_training_vec_env(env_config: EnvConfig, n_envs: int, training: bool,
                           norm_reward: bool, clip_obs: float = 10.0):
    """Egitim/EvalCallback icin Monitor + DummyVecEnv + VecNormalize sarilmis
    ortam. train.py'nin hem egitim hem eval_env'i icin kullanilir.

    training=True  + norm_reward=True  -> egitim ortami
    training=False + norm_reward=False -> EvalCallback icin eval ortami
    """
    def _make():
        return Monitor(make_env(env_config))

    venv = DummyVecEnv([_make for _ in range(n_envs)])
    venv = VecNormalize(
        venv, norm_obs=True, norm_reward=norm_reward, clip_obs=clip_obs, training=training
    )
    return venv


def make_eval_vec_env(env_config: EnvConfig):
    """evaluate.py icin: TEK, Monitor'SUZ F450HoverEnv iceren DummyVecEnv.

    Monitor sarilmamasi bilincli bir tercih: evaluate.py, venv.envs[0]
    uzerinden dogrudan .fdm'e erisip JSBSim property'lerini okuyor.
    Monitor sarsaydi venv.envs[0] bir Monitor nesnesi olur, .fdm
    bulunamazdi (AttributeError).
    """
    return DummyVecEnv([lambda: make_env(env_config)])

In [ ]:
%%bash
# Repo dizinine geç
cd /content/repo

# Tüm değişiklikleri ekle
git add -A

# Değişiklikleri açıkla (Eğer içeride değişiklik varsa)
if ! git diff-index --quiet HEAD --; then
  git commit -m "ACMI"
  git push origin main
  echo "Değişiklikler başarıyla pushlandı."
else
  echo "Pushlanacak bir değişiklik bulunamadı."
fi

[main f4a7acc] Güncel telemetry ve değerlendirme değişiklikleri
 3 files changed, 36 insertions(+), 37 deletions(-)
Değişiklikler başarıyla pushlandı.


To https://github.com/miray7yuce/quadcopter-rl-copilot.git
   fbb7bfa..f4a7acc  main -> main


In [ ]:
%%bash
cd /content/repo
git add -A
if ! git diff-index --quiet HEAD --; then
  git commit -m "Added EvalCallback to training pipeline and updated configs"
  git push origin main
  echo "Tüm değişiklikler başarıyla GitHub'a gönderildi."
else
  echo "Pushlanacak yeni bir değişiklik bulunamadı."
fi

[main be8e683] Added EvalCallback to training pipeline and updated configs
 3 files changed, 100 insertions(+), 23 deletions(-)
Tüm değişiklikler başarıyla GitHub'a gönderildi.


To https://github.com/miray7yuce/quadcopter-rl-copilot.git
   f4a7acc..be8e683  main -> main
